# Final Submission Notebook

This single notebook consolidates all three project parts (search, optimization, and learning) with runnable code, test cases, and recorded results.

Key data notes:
- Parts 1 and 2 use demo datasets because the official assignment datasets are not tracked in this repo.
- Part 3 trains and evaluates on the real GTSRB dataset at [gtsrb/Train](gtsrb/Train).

Recorded Part 3 result (real GTSRB):
- Images loaded: 39,209
- Held-out test split: 7,842
- Epochs: 6, batch size: 32
- Accuracy: 0.9872 (7,742 / 7,842 correct)
- Saved model: [reports/artifacts/gtsrb_model.keras](reports/artifacts/gtsrb_model.keras)

## Setup and paths

The next cell sets shared paths, helper utilities, and quick sanity checks. It reads the canonical metrics file at [reports/evaluation_metrics.json](reports/evaluation_metrics.json) and uses figures in [reports/figures](reports/figures).

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
METRICS_PATH = REPORTS_DIR / "evaluation_metrics.json"
MODEL_PATH = REPORTS_DIR / "artifacts" / "gtsrb_model.keras"
GTSRB_DIR = PROJECT_ROOT / "gtsrb" / "Train"

def resolve_path(path_str):
    return PROJECT_ROOT / Path(path_str)

def show_image(path, title=None, figsize=(8, 5)):
    image = mpimg.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

print("Project root:", PROJECT_ROOT)
print("GTSRB dir exists:", GTSRB_DIR.is_dir())
print("Metrics file:", METRICS_PATH)
print("Model artifact:", MODEL_PATH)

## Part 1 - Flight Connections (BFS)

Implementation: [src/part1_search/flights.py](src/part1_search/flights.py)

Test cases (demo flight graph):

| Case | Expected hops | Actual hops | Passed |
| --- | --- | --- | --- |
| Windhoek to Cairo | 3 | 3 | True |
| Johannesburg to Lagos | 1 | 1 | True |
| Nairobi to Nairobi | 0 | 0 | True |
| Cairo to Windhoek | None | None | True |

Figure: [reports/figures/part1_route_lengths.png](reports/figures/part1_route_lengths.png)

In [ ]:
from src.evaluation.run_evaluation import run_part1

part1 = run_part1()
part1_results = pd.DataFrame(part1["results"])
part1_results[["case", "expected_hops", "actual_hops", "passed"]]

show_image(resolve_path(part1["figure"]), "Part 1 route lengths")

## Part 2 - Hospital Shift Scheduler (CSP)

Implementation: [src/part2_optimization/run_scheduler.py](src/part2_optimization/run_scheduler.py) and [src/models/Csp.py](src/models/Csp.py)

Demo metrics:
- All 21 shifts assigned, no leave violations, no rest-rule violations.
- Max shifts per nurse: 5
- Fairness standard deviation: 1.6

Shift counts (demo):

| Nurse | Shifts |
| --- | --- |
| Alice | 5 |
| Bob | 5 |
| Carol | 5 |
| David | 5 |
| Eve | 1 |

Figures:
- [reports/figures/part2_shift_distribution.png](reports/figures/part2_shift_distribution.png)
- [reports/figures/part2_schedule_overview.png](reports/figures/part2_schedule_overview.png)

In [ ]:
from src.evaluation.run_evaluation import run_part2

part2 = run_part2()
metrics = part2["metrics"]

pd.DataFrame([metrics])[
    ["complete", "fully_assigned", "passes_constraints", "max_shifts_per_nurse", "fairness_stddev"]
].rename(columns={"max_shifts_per_nurse": "max_shifts"})

shift_counts = pd.DataFrame(
    [{"nurse": nurse, "shifts": count} for nurse, count in metrics["shift_counts"].items()]
).sort_values("nurse")
shift_counts

for fig in part2["figures"]:
    title = f"Part 2 - {Path(fig).stem.replace('_', ' ').title()}"
    show_image(resolve_path(fig), title)

## Part 3 - Traffic Sign Recognition (CNN)

Implementation: [src/train.py](src/train.py) and [src/models/Cnn.py](src/models/Cnn.py)

Recorded evaluation (real GTSRB):
- Dataset: [gtsrb/Train](gtsrb/Train)
- Images loaded: 39,209
- Held-out test split: 7,842
- Epochs: 6
- Batch size: 32
- Accuracy: 0.9872
- Correct predictions: 7,742 / 7,842
- Model artifact: [reports/artifacts/gtsrb_model.keras](reports/artifacts/gtsrb_model.keras)

Figures:
- [reports/figures/confusion.png](reports/figures/confusion.png)
- [reports/figures/sample_predictions.png](reports/figures/sample_predictions.png)
- [reports/figures/training_curve.png](reports/figures/training_curve.png)

In [ ]:
if not GTSRB_DIR.is_dir():
    raise FileNotFoundError("Expected GTSRB dataset at gtsrb/Train")

with METRICS_PATH.open("r", encoding="utf-8") as handle:
    metrics = json.load(handle)

part3 = metrics["part3"]
part3_summary = pd.DataFrame(
    [
        {
            "dataset": part3["dataset"],
            "total_images": part3["total_images"],
            "test_images": part3["test_images"],
            "epochs": part3["epochs"],
            "batch_size": part3["batch_size"],
            "accuracy": round(part3["accuracy"], 4),
            "correct_predictions": f"{part3['correct_predictions']} / {part3['test_images']}",
            "model_artifact": part3["model_artifact"],
        }
    ]
)
part3_summary

for fig in part3["figures"]:
    title = f"Part 3 - {Path(fig).stem.replace('_', ' ').title()}"
    show_image(resolve_path(fig), title)

# Optional rerun (will retrain):
# from src.evaluation.run_evaluation import run_part3
# os.environ["GTSRB_DIR"] = str(GTSRB_DIR)
# part3_run = run_part3()
# part3_run

## Report and reference artifacts

The notebook aligns with the final report artifacts:

- Background and theory: [reports/background.md](reports/background.md)
- Literature summary: [references/literature-summary.md](references/literature-summary.md)
- Results summary: [reports/results.md](reports/results.md)
- Evaluation notes: [reports/evaluation_notes.md](reports/evaluation_notes.md)
- Canonical metrics: [reports/evaluation_metrics.json](reports/evaluation_metrics.json)
- Slide storyline: [presentation/slide_draft.md](presentation/slide_draft.md)

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# Load evaluation metrics
metrics_path = '../reports/evaluation_metrics.json'
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

print("✓ Loaded evaluation metrics from reports/evaluation_metrics.json")

✓ Loaded evaluation metrics from reports/evaluation_metrics.json


In [2]:
# Part 1 Results Summary
print("=" * 60)
print("PART 1 - FLIGHT CONNECTIONS (BFS Search)")
print("=" * 60)

part1 = metrics['part1']
print(f"\nDataset: {part1['dataset']}")
print(f"Test Cases: {part1['cases_passed']}/{part1['cases_run']} passed ✓")
print(f"\nTest Results:")
print("-" * 60)

for result in part1['results']:
    status = "✓" if result['passed'] else "✗"
    print(f"{status} {result['case']}: {result['hop_count']} hops" if result['path_found'] else f"{status} {result['case']}: No path found")

print(f"\nFigure: {part1['figure']}")
print("Implementation: src/part1_search/flights.py")

PART 1 - FLIGHT CONNECTIONS (BFS Search)

Dataset: data\demo_flights
Test Cases: 4/4 passed ✓

Test Results:
------------------------------------------------------------
✓ Windhoek to Cairo: 3 hops
✓ Johannesburg to Lagos: 1 hops
✓ Nairobi to Nairobi: 0 hops
✓ Cairo to Windhoek: No path found

Figure: reports\figures\part1_route_lengths.png
Implementation: src/part1_search/flights.py


In [ ]:
# Part 2 Results Summary
print("\n" + "=" * 60)
print("PART 2 - HOSPITAL SHIFT SCHEDULING (CSP Solver)")
print("=" * 60)

part2 = metrics['part2']
metrics_p2 = part2['metrics']
print(f"\nDataset: {part2['dataset']}")
print(f"Total Nurses: {part2['nurses']}")
print(f"Total Shifts: 21 (7 days × 3 shifts)")
print(f"\nOptimization Results:")
print(f"  • All Shifts Assigned: {metrics_p2['fully_assigned']} ✓")
print(f"  • Leave Violations: {len(metrics_p2['leave_violations'])} ✓")
print(f"  • Rest Rule Violations: {len(metrics_p2['rest_violations'])} ✓")
print(f"  • Max Shifts per Nurse: {metrics_p2['max_shifts_per_nurse']}")
print(f"  • Fairness (StdDev): {metrics_p2['fairness_stddev']:.1f}")

print(f"\nFigures:")
for fig in part2['figures']:
    print(f"  • {fig}")
print("Implementation: src/part2_optimization/run_scheduler.py, src/models/Csp.py")


PART 2 - HOSPITAL SHIFT SCHEDULING (CSP Solver)

Dataset: data\demo_staff\staff_small.txt
Total Shifts: 21 (7 days × 3 shifts)

Optimization Results:


KeyError: 'shifts_assigned'

In [ ]:
# Part 3 Results Summary
print("\n" + "=" * 60)
print("PART 3 - TRAFFIC SIGN RECOGNITION (CNN Classification)")
print("=" * 60)

part3 = metrics['part3']
print(f"\nDataset: {part3['dataset']}")
print(f"Total Images: {part3['total_images']}")
print(f"Test Samples: {part3['test_images']}")
print(f"Classes: 43 (German Traffic Signs)")

print(f"\nTraining Configuration:")
print(f"  • Epochs: {part3['epochs']}")
print(f"  • Batch Size: {part3['batch_size']}")
print(f"  • Loss Function: Categorical Crossentropy")
print(f"  • Optimizer: Adam")

print(f"\n★ TEST ACCURACY: {part3['accuracy']:.4f} ({part3['correct_predictions']}/{part3['test_images']} correct) ★")
print(f"  • Test Loss: {part3['loss']:.4f}")
print(f"  • Confusion Matrix Shape: {part3['confusion_matrix_validation']['shape']}")

print(f"\nModel Artifact: {part3['model_artifact']}")
print(f"Figures:")
for fig in part3['figures']:
    print(f"  • {fig}")
print("Implementation: src/train.py, src/part3_ml/train.py, src/models/Cnn.py")

In [ ]:
# Display all evaluation figures
print("\n" + "=" * 60)
print("EVALUATION FIGURES")
print("=" * 60)

figures_dir = '../reports/figures'
figure_files = [f for f in os.listdir(figures_dir) if f.endswith('.png')]
figure_files.sort()

print(f"\nGenerated {len(figure_files)} figures:\n")

# Create a 3x2 subplot layout
n_figures = len(figure_files)
n_rows = (n_figures + 2) // 3
n_cols = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for idx, img_file in enumerate(figure_files):
    img_path = os.path.join(figures_dir, img_file)
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(img_file, fontsize=10, fontweight='bold')
    axes[idx].axis('off')
    print(f"  {idx + 1}. {img_file}")

# Hide unused subplots
for idx in range(len(figure_files), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/submission_figures_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Generated submission_figures_grid.png")

In [ ]:
# Final Summary Table
print("\n" + "=" * 60)
print("FINAL PROJECT SUMMARY")
print("=" * 60)

summary_data = {
    'Task': ['Part 1: Flight Search', 'Part 2: Shift Scheduling', 'Part 3: Traffic Signs'],
    'Method': ['BFS Graph Search', 'CSP with AC-3', 'Convolutional Neural Network'],
    'Status': ['✓ Complete', '✓ Complete', '✓ Complete'],
    'Key Result': [
        '4/4 test cases passed',
        'All 21 shifts assigned',
        '98.72% accuracy (7,742/7,842)'
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

print("\n" + "=" * 60)
print("PROJECT COMPLETION STATUS")
print("=" * 60)
print("""
✓ Background theory documented (reports/background.md)
✓ Literature review completed (references/literature-summary.md)
✓ All three parts implemented and tested
✓ Evaluation metrics computed (reports/evaluation_metrics.json)
✓ Figures generated and validated
✓ Model artifact saved (reports/artifacts/gtsrb_model.keras)
✓ Results summary available (reports/results.md)

Ready for final submission!
""")

print("=" * 60)
print("REFERENCE DOCUMENTS")
print("=" * 60)
print(f"""
• Background: reports/background.md
• Literature: references/literature-summary.md
• Results: reports/results.md
• Metrics: reports/evaluation_metrics.json
• Evaluation Notes: reports/evaluation_notes.md
• Presentation Outline: presentation/slide_draft.md
""")